# Huấn luyện AI nhận diện vật thể cho ScreenWatcher
Chạy lần lượt từng ô (bấm ▶). Cần: file `dataset_coco.zip` xuất từ app (🏷 Gán nhãn ảnh → 📤 Xuất COCO).
Nên bật GPU: **Runtime → Change runtime type → T4 GPU**. Cuối cùng tải về `model.tflite`, rồi nạp trong app (📥 Nạp mô hình).

## 1. Cài thư viện (mất vài phút)
MediaPipe Model Maker cần Python 3.9–3.12. Nếu ô này báo lỗi, hãy chụp lại thông báo lỗi.

In [ ]:
!python --version
!pip install --upgrade pip
!pip install mediapipe-model-maker

## 2. Tải bộ dữ liệu lên
Bấm *Choose files* và chọn `dataset_coco.zip` (đưa file từ điện thoại lên Google Drive/tải thẳng đều được).

In [ ]:
from google.colab import files
up = files.upload()
!rm -rf dataset && mkdir dataset
!unzip -q -o dataset_coco.zip -d dataset
!ls dataset dataset/train | head

## 3. Nạp dữ liệu

In [ ]:
import os, json
import tensorflow as tf
assert tf.__version__.startswith('2')
from mediapipe_model_maker import object_detector

train_data = object_detector.Dataset.from_coco_folder("dataset/train", cache_dir="/tmp/od_data/train")
validation_data = object_detector.Dataset.from_coco_folder("dataset/validation", cache_dir="/tmp/od_data/validation")
print("Ảnh huấn luyện:", train_data.size, "| ảnh kiểm tra:", validation_data.size)
print("Các loại:", [c["name"] for c in json.load(open("dataset/train/labels.json"))["categories"]])

## 4. Huấn luyện
Tăng `EPOCHS` (30–100) nếu kết quả chưa tốt; ít ảnh thì tăng thêm ảnh sẽ hiệu quả hơn tăng EPOCHS.

In [ ]:
EPOCHS = 50
spec = object_detector.SupportedModels.MOBILENET_MULTI_AVG
hparams = object_detector.HParams(export_dir='exported_model', epochs=EPOCHS, batch_size=8)
options = object_detector.ObjectDetectorOptions(supported_model=spec, hparams=hparams)
model = object_detector.ObjectDetector.create(
    train_data=train_data, validation_data=validation_data, options=options)

## 5. Đánh giá

In [ ]:
loss, coco_metrics = model.evaluate(validation_data, batch_size=4)
print("Loss:", loss)
print("COCO metrics:", coco_metrics)

## 6. Xuất và tải mô hình về
Tải xong, chép `model.tflite` vào điện thoại rồi trong app bấm **📥 Nạp mô hình**.

In [ ]:
model.export_model()
!ls -la exported_model
files.download('exported_model/model.tflite')